In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [2]:
def preprocessing_Timestamp(df):
    # Convert TIMESTAMP to datetime
    df["TIMESTAMP"] = pd.to_datetime(df["TIMESTAMP"])
    
    # Extract date components
    df["Month"] = df["TIMESTAMP"].dt.month
    df["Day"] = df["TIMESTAMP"].dt.day
    df["Hour"] = df["TIMESTAMP"].dt.hour
    df["Minute"] = df["TIMESTAMP"].dt.minute

    df = df.drop(["TIMESTAMP"], axis = 1) # TIMESTAMP 열 삭제
    
    return df

train = preprocessing_Timestamp(train)
test = preprocessing_Timestamp(test)

In [3]:
from sklearn.preprocessing import LabelEncoder

def preprocessing_Line_Product(df):
    # 'LINE'과 'PRODUCT_CODE_encoded'의 조합을 하나의 문자열로 결합
    df['LINE_PRODUCT_COMBINATION'] = df['LINE'].astype(str) + '_' + df['PRODUCT_CODE'].astype(str)

    # LabelEncoder를 사용하여 고유한 숫자 레이블 부여
    label_encoder = LabelEncoder()
    df['LINE_PRODUCT_LABEL'] = label_encoder.fit_transform(df['LINE_PRODUCT_COMBINATION'])

    df = df.drop(["LINE", "PRODUCT_CODE", "LINE_PRODUCT_COMBINATION"], axis = 1)

    return df

train = preprocessing_Line_Product(train)
test = preprocessing_Line_Product(test)

In [4]:
train_x = train.drop(columns=['PRODUCT_ID', 'Y_Quality'])
train_y = train['Y_Class']

test_x = test.drop(columns=['PRODUCT_ID'])

In [5]:
from sklearn.preprocessing import MinMaxScaler

# X 컬럼 추출
x_cols = train_x.columns[train_x.columns.str.startswith('X')].tolist()

# 그룹 매핑 함수
def map_line_group(label):
    if label in [0, 1]:
        return 0
    elif label in [2, 3]:
        return 1
    elif label in [4, 6]:
        return 2
    elif label in [5, 7]:
        return 3
    else:
        return -1

# 학습 데이터에 그룹 라벨 추가
train_x['LINE_GROUP_LABEL'] = train_x['LINE_PRODUCT_LABEL'].apply(map_line_group)

# 그룹별 스케일러 저장 딕셔너리
group_scalers = {}

# 그룹별 정규화 및 스케일러 저장
for group_label in train_x['LINE_GROUP_LABEL'].unique():
    idx = train_x['LINE_GROUP_LABEL'] == group_label
    scaler = MinMaxScaler()
    train_x.loc[idx, x_cols] = scaler.fit_transform(train_x.loc[idx, x_cols])
    group_scalers[group_label] = scaler  # 저장

# 보조 컬럼 제거
train_x.drop(columns=['LINE_GROUP_LABEL'], inplace=True)

# 테스트 데이터에도 그룹 라벨 생성
test_x['LINE_GROUP_LABEL'] = test_x['LINE_PRODUCT_LABEL'].apply(map_line_group)

# 그룹별 정규화 적용
for group_label in test_x['LINE_GROUP_LABEL'].unique():
    idx = test_x['LINE_GROUP_LABEL'] == group_label
    scaler = group_scalers[group_label]  # 학습된 스케일러 가져오기
    test_x.loc[idx, x_cols] = scaler.transform(test_x.loc[idx, x_cols])

# 필요 시 제거
test_x.drop(columns=['LINE_GROUP_LABEL'], inplace=True)

c:\Users\twoh0\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\_array_api.py:776: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmin(X, axis=axis))
c:\Users\twoh0\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\_array_api.py:793: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmax(X, axis=axis))
c:\Users\twoh0\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\_array_api.py:776: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmin(X, axis=axis))
c:\Users\twoh0\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\_array_api.py:793: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmax(X, axis=axis))
c:\Users\twoh0\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\_array_api.py:776: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmin(X, axis=axis))
c:\Users\twoh0\AppDa

In [6]:
#빈 slot을 모두 평균으로 채움
train_x = train_x.fillna(train_x.mean())
test_x = test_x.fillna(train_x.mean())

train_x = train_x.dropna(axis = 1)
test_x = test_x.dropna(axis = 1)

In [7]:
# X로 시작하는 수치형 컬럼 선택
x_cols = train_x.columns[train_x.columns.str.startswith('X')].tolist()

# 상관관계 계산
corr = train_x[x_cols + ['Y_Class']].corr()

# Y_Class 기준 상관계수 정렬 (상위 몇 개만 보기 원할 수도 있음)
corr_with_target = corr['Y_Class'].drop('Y_Class').sort_values(key = abs, ascending=False)

top_features = [i for i in corr_with_target.index if abs(corr_with_target[i]) > 0.05]
top_features_sorted = [col for col in train_x.columns if col in top_features]

train_x = train_x[top_features_sorted + ['Month', 'Day', 'Hour', 'Minute', 'LINE_PRODUCT_LABEL']]
test_x = test_x[top_features_sorted + ['Month', 'Day', 'Hour', 'Minute', 'LINE_PRODUCT_LABEL']] 

train_x
# test_x

,X_2,X_5,X_8,X_24,X_38,X_44,X_56,X_62,X_73,X_90,...,X_2865,X_2866,X_2867,X_2869,X_2870,Month,Day,Hour,Minute,LINE_PRODUCT_LABEL
0,0.542605,0.39255,0.048711,0.13467,0.590974,0.406615,0.505571,0.497577,0.507610,0.526743,...,0.550000,0.256757,0.248647,0.122283,0.890487,6,13,5,14,2
1,0.542605,0.39255,0.048711,0.13467,0.590974,0.406615,0.505571,0.497577,0.507610,0.526743,...,0.550000,0.240754,0.300866,0.164742,0.601770,6,13,5,22,3
2,0.542605,0.39255,0.048711,0.13467,0.590974,0.406615,0.505571,0.497577,0.507610,0.526743,...,0.550000,0.251422,0.133929,0.205163,0.922566,6,13,5,30,2
3,0.542605,0.39255,0.048711,0.13467,0.590974,0.406615,0.505571,0.497577,0.507610,0.526743,...,0.550000,0.199858,0.202110,0.003057,0.559181,6,13,5,39,3
4,0.542605,0.39255,0.048711,0.13467,0.590974,0.406615,0.505571,0.497577,0.507610,0.526743,...,0.500000,0.233997,0.275703,0.088315,0.846239,6,13,5,47,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
593,0.533333,0.00000,0.000000,0.00000,0.000000,0.530612,0.644444,0.570093,0.641975,0.500000,...,0.589759,0.664555,0.592741,0.719083,0.275426,9,8,14,30,7
594,0.542605,0.39255,0.048711,0.13467,0.590974,0.406615,0.505571,0.497577,0.507610,0.526743,...,0.550000,0.616999,0.578193,0.835938,0.266593,9,8,22,38,2
595,0.542605,0.39255,0.048711,0.13467,0.590974,0.406615,0.505571,0.497577,0.507610,0.526743,...,0.500000,0.664555,0.592741,0.719083,0.275426,9,8,22,47,2
596,0.538462,1.00000,0.000000,0.00000,0.750000,0.173913,0.000000,0.131148,1.000000,0.000000,...,0.589759,0.664555,0.592741,0.719083,0.275426,9,8,14,38,4


In [9]:
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score# 정확도 함수
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(train_x, train_y, test_size=0.2, random_state = 100)

# params = {
#     'num_leaves': [70, 80, 90, 100, 110, 120, 130],
#     'learning_rate': [0.05, 0.1],
#     'n_estimators': [70, 80, 90, 100, 110, 120, 130],
# }
# grid = GridSearchCV(lgbm.LGBMClassifier(random_state=37), params, scoring='f1_macro', cv=3)
# grid.fit(train_x, train_y)
# print(grid.best_params_)


clf = RandomForestClassifier(random_state = 100)
clf.fit(x_train, y_train)
clf.predict(x_test)

y_pred = clf.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')

print(f"Accuracy: {accuracy:.4f}")
print(f"Macro F1 Score: {macro_f1:.4f}")

Y_Label = clf.predict(test_x)
submit = pd.read_csv("sample_submission.csv")
submit['Y_Class'] = Y_Label
submit.to_csv("submission.csv", index=False)

Accuracy: 0.8167
Macro F1 Score: 0.7017
